# Adversarial Robustness

This notebook evaluates how robust the trained colorization model is to adversarial
perturbations on the input grayscale channel, and trains a version with adversarial
augmentation to improve robustness.

Two attack methods are used:
- **FGSM** (Fast Gradient Sign Method): single-step attack
- **PGD** (Projected Gradient Descent): iterative attack, stronger

Perturbations are applied to the L channel before inference. We measure PSNR and
SSIM degradation as epsilon increases, then compare a standard model against one
trained with adversarial examples injected during Stage 2.


In [ ]:
!pip install fastai>=2.7 scikit-image tqdm -q


In [ ]:
import os
import glob
import numpy as np
from PIL import Image
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
from skimage.color import rgb2lab, lab2rgb

import torch
from torch import nn, optim
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader

from fastai.vision.learner import create_body
from torchvision.models.resnet import resnet18
from fastai.vision.models.unet import DynamicUnet

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
SIZE = 256
BASELINE_CHECKPOINT_DIR = "/content/drive/MyDrive/colorization_checkpoints"


def build_res_unet(n_input=1, n_output=2, size=256, dropout=0.5):
    try:
        from torchvision.models import ResNet18_Weights
        model = resnet18(weights='DEFAULT')
    except:
        try:
            model = resnet18(pretrained=True)
        except:
            model = resnet18(pretrained=False)
            print("warning: could not load pretrained weights")

    if n_input == 1:
        old_conv = model.conv1
        with torch.no_grad():
            new_conv = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
            new_conv.weight.data = old_conv.weight.data.mean(dim=1, keepdim=True)
        model.conv1 = new_conv

    body = create_body(model, cut=-2)
    return DynamicUnet(body, n_output, (size, size)).to(
        torch.device("cuda" if torch.cuda.is_available() else "cpu"))


class PatchDiscriminator(nn.Module):
    def __init__(self, input_c, num_filters=64, n_down=3):
        super().__init__()
        self.model = self.get_layers(input_c, num_filters, n_down)

    def get_layers(self, input_c, num_filters, n_down):
        model = [self.get_conv(input_c, num_filters, norm=False)]
        for i in range(n_down):
            model += [self.get_conv(num_filters * 2**i, num_filters * 2**(i+1),
                                    stride=1 if i == (n_down - 1) else 2)]
        model += [self.get_conv(num_filters * 2**n_down, 1, stride=1, norm=False, act=False)]
        return nn.Sequential(*model)

    def get_conv(self, in_c, out_c, kernel_size=4, stride=2, padding=1, norm=True, act=True):
        layers = [nn.Conv2d(in_c, out_c, kernel_size, stride, padding, bias=not norm)]
        if norm: layers.append(nn.BatchNorm2d(out_c))
        if act:  layers.append(nn.LeakyReLU(0.2, True))
        return nn.Sequential(*layers)

    def forward(self, x):
        return self.model(x)


class GANLoss(nn.Module):
    def __init__(self, gan_mode='vanilla', real_label=1.0, fake_label=0.0):
        super().__init__()
        self.register_buffer('real_label', torch.tensor(real_label))
        self.register_buffer('fake_label', torch.tensor(fake_label))
        self.loss = nn.BCEWithLogitsLoss() if gan_mode == 'vanilla' else nn.MSELoss()

    def get_labels(self, preds, target_is_real):
        labels = self.real_label if target_is_real else self.fake_label
        return labels.expand_as(preds)

    def __call__(self, preds, target_is_real):
        return self.loss(preds, self.get_labels(preds, target_is_real))


def init_weights(net, gain=0.02):
    def init_func(m):
        classname = m.__class__.__name__
        if hasattr(m, 'weight') and 'Conv' in classname:
            nn.init.normal_(m.weight.data, mean=0.0, std=gain)
            if hasattr(m, 'bias') and m.bias is not None:
                nn.init.constant_(m.bias.data, 0.0)
        elif 'BatchNorm2d' in classname:
            nn.init.normal_(m.weight.data, 1., gain)
            nn.init.constant_(m.bias.data, 0.)
    net.apply(init_func)
    return net


def init_model(model, device):
    return init_weights(model.to(device))


def total_variation_loss(img):
    diff_h = torch.abs(img[:, :, :, :-1] - img[:, :, :, 1:])
    diff_v = torch.abs(img[:, :, :-1, :] - img[:, :, 1:, :])
    return diff_h.mean() + diff_v.mean()


def contrast_loss(fake_img, real_img):
    fake_L = fake_img[:, 0:1, :, :]
    real_L = real_img[:, 0:1, :, :]
    fake_std = torch.std(fake_L.view(fake_L.size(0), -1), dim=1)
    real_std = torch.std(real_L.view(real_L.size(0), -1), dim=1)
    return torch.abs(fake_std - real_std).mean()


class AverageMeter:
    def __init__(self):
        self.reset()
    def reset(self):
        self.count, self.avg, self.sum = [0.] * 3
    def update(self, val, count=1):
        self.count += count
        self.sum  += count * val
        self.avg   = self.sum / self.count


def lab_to_rgb(L, ab):
    L  = (L + 1.) * 50.
    ab = ab * 110.
    Lab = torch.cat([L, ab], dim=1).permute(0, 2, 3, 1).cpu().numpy()
    return np.stack([lab2rgb(img) for img in Lab], axis=0)


def visualize(model, data, save=False, path="result.png"):
    model.net_G.eval()
    with torch.no_grad():
        model.setup_input(data)
        model.forward()
    model.net_G.train()
    fake = lab_to_rgb(model.L, model.fake_color.detach())
    real = lab_to_rgb(model.L, model.ab)
    n = min(5, len(model.L))
    fig, axes = plt.subplots(3, n, figsize=(3*n, 9))
    for i in range(n):
        axes[0, i].imshow(model.L[i][0].cpu(), cmap='gray'); axes[0, i].axis('off')
        axes[1, i].imshow(fake[i]);                           axes[1, i].axis('off')
        axes[2, i].imshow(real[i]);                           axes[2, i].axis('off')
    axes[0, 0].set_ylabel("input");       axes[1, 0].set_ylabel("output")
    axes[2, 0].set_ylabel("ground truth")
    plt.tight_layout()
    if save: plt.savefig(path, bbox_inches='tight')
    plt.show()


print("definitions loaded")


## Attack Functions


In [ ]:
def fgsm_attack(model, L, ab, epsilon):
    L_adv = L.clone().detach().requires_grad_(True)
    fake_color = model.net_G(L_adv)
    loss = nn.L1Loss()(fake_color, ab)
    loss.backward()
    perturbation = epsilon * L_adv.grad.sign()
    return torch.clamp(L + perturbation, -1.0, 1.0).detach()


def pgd_attack(model, L, ab, epsilon, alpha=0.01, num_steps=10):
    L_adv = L.clone().detach()
    for _ in range(num_steps):
        L_adv = L_adv.requires_grad_(True)
        fake_color = model.net_G(L_adv)
        loss = nn.L1Loss()(fake_color, ab)
        loss.backward()
        L_adv = (L_adv + alpha * L_adv.grad.sign()).detach()
        L_adv = torch.clamp(L_adv, L - epsilon, L + epsilon)
        L_adv = torch.clamp(L_adv, -1.0, 1.0)
    return L_adv


In [ ]:
def compute_psnr(pred, target, max_val=1.0):
    mse = torch.mean((pred - target) ** 2)
    if mse == 0:
        return float('inf')
    return 20 * torch.log10(torch.tensor(max_val) / torch.sqrt(mse))


def compute_ssim_batch(pred, target):
    # simple luminance-based SSIM approximation
    mu_x = pred.mean()
    mu_y = target.mean()
    sigma_x  = pred.std()
    sigma_y  = target.std()
    sigma_xy = ((pred - mu_x) * (target - mu_y)).mean()
    c1, c2 = 0.01**2, 0.03**2
    return float(((2 * mu_x * mu_y + c1) * (2 * sigma_xy + c2)) /
                 ((mu_x**2 + mu_y**2 + c1) * (sigma_x**2 + sigma_y**2 + c2)))


In [ ]:
DATASET_PATH = "/content/drive/MyDrive/datasets/coco_subset_16000"
NUM_IMAGES   = 13000


class ColorizationDataset(Dataset):
    def __init__(self, paths, split='train'):
        self.paths = paths
        self.split = split
        if split == 'train':
            self.transforms = transforms.Compose([
                transforms.Resize((256, 256), Image.BICUBIC),
                transforms.RandomHorizontalFlip(),
            ])
        else:
            self.transforms = transforms.Resize((256, 256), Image.BICUBIC)

    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert("RGB")
        img = self.transforms(img)
        img_lab = rgb2lab(np.array(img)).astype("float32")
        img_lab = transforms.ToTensor()(img_lab)
        L  = img_lab[[0], ...] / 50. - 1.
        ab = img_lab[[1, 2], ...] / 110.
        return {'L': L, 'ab': ab}

    def __len__(self):
        return len(self.paths)


def make_dataloaders(paths, split='train', batch_size=16, n_workers=2):
    return DataLoader(ColorizationDataset(paths, split), batch_size=batch_size,
                      num_workers=n_workers, pin_memory=True, shuffle=(split == 'train'))


# load paths
image_extensions = ['*.jpg', '*.jpeg', '*.png', '*.JPG', '*.JPEG', '*.PNG']
paths = []
for ext in image_extensions:
    paths.extend(glob.glob(os.path.join(DATASET_PATH, ext)))
    paths.extend(glob.glob(os.path.join(DATASET_PATH, '**', ext), recursive=True))

if not paths:
    raise RuntimeError(f"no images found in {DATASET_PATH}")

np.random.seed(123)
if len(paths) > NUM_IMAGES:
    paths = np.random.choice(paths, NUM_IMAGES, replace=False)

rand_idxs   = np.random.permutation(len(paths))
train_paths = paths[rand_idxs[:int(len(paths) * 0.8)]]
val_paths   = paths[rand_idxs[int(len(paths) * 0.8):]]

BATCH_SIZE  = 16
NUM_WORKERS = 2

train_dl = make_dataloaders(train_paths, split='train', batch_size=BATCH_SIZE, n_workers=NUM_WORKERS)
val_dl   = make_dataloaders(val_paths,   split='val',   batch_size=BATCH_SIZE, n_workers=NUM_WORKERS)

print(f"train: {len(train_paths)}  val: {len(val_paths)}")
print(f"train batches: {len(train_dl)}  val batches: {len(val_dl)}")


## Load Baseline Model


In [ ]:
class MainModel(nn.Module):
    def __init__(self, net_G=None, lr_G=2e-4, lr_D=2e-4,
                 beta1=0.5, beta2=0.999, lambda_L1=100., lambda_TV=1.0, lambda_contrast=0.0):
        super().__init__()
        self.device          = device
        self.lambda_L1       = lambda_L1
        self.lambda_TV       = lambda_TV
        self.lambda_contrast = lambda_contrast

        self.net_G = net_G.to(self.device) if net_G else                      init_model(build_res_unet(n_input=1, n_output=2, size=SIZE), self.device)
        self.net_D        = init_model(PatchDiscriminator(input_c=3, n_down=3, num_filters=64), self.device)
        self.GANcriterion = GANLoss(gan_mode='vanilla').to(self.device)
        self.L1criterion  = nn.L1Loss()
        self.opt_G = optim.Adam(self.net_G.parameters(), lr=lr_G, betas=(beta1, beta2))
        self.opt_D = optim.Adam(self.net_D.parameters(), lr=lr_D, betas=(beta1, beta2))

    def set_requires_grad(self, model, requires_grad=True):
        for p in model.parameters():
            p.requires_grad = requires_grad

    def setup_input(self, data):
        self.L  = data['L'].to(self.device)
        self.ab = data['ab'].to(self.device)

    def forward(self):
        self.fake_color = self.net_G(self.L)

    def backward_D(self):
        fake_preds = self.net_D(torch.cat([self.L, self.fake_color], dim=1).detach())
        self.loss_D_fake = self.GANcriterion(fake_preds, False)
        real_preds = self.net_D(torch.cat([self.L, self.ab], dim=1))
        self.loss_D_real = self.GANcriterion(real_preds, True)
        self.loss_D = (self.loss_D_fake + self.loss_D_real) * 0.5
        self.loss_D.backward()

    def backward_G(self):
        fake_image = torch.cat([self.L, self.fake_color], dim=1)
        real_image = torch.cat([self.L, self.ab], dim=1)
        fake_preds = self.net_D(fake_image)
        self.loss_G_GAN      = self.GANcriterion(fake_preds, True)
        self.loss_G_L1       = self.L1criterion(self.fake_color, self.ab) * self.lambda_L1
        self.loss_G_TV       = total_variation_loss(fake_image) * self.lambda_TV
        if self.lambda_contrast > 0:
            self.loss_G_contrast = contrast_loss(fake_image, real_image) * self.lambda_contrast
        else:
            self.loss_G_contrast = torch.tensor(0.0, device=self.device)
        self.loss_G = self.loss_G_GAN + self.loss_G_L1 + self.loss_G_TV + self.loss_G_contrast
        self.loss_G.backward()

    def optimize(self):
        self.forward()
        self.net_D.train(); self.set_requires_grad(self.net_D, True)
        self.opt_D.zero_grad(); self.backward_D(); self.opt_D.step()
        self.net_G.train(); self.set_requires_grad(self.net_D, False)
        self.opt_G.zero_grad(); self.backward_G(); self.opt_G.step()


In [ ]:
ADV_CHECKPOINT_DIR = "/content/drive/MyDrive/colorization_checkpoints_adversarial"
os.makedirs(ADV_CHECKPOINT_DIR, exist_ok=True)

net_G = build_res_unet(n_input=1, n_output=2, size=SIZE)

# try to load the final baseline model first, fall back to pretrained generator
for candidate in [
    os.path.join(BASELINE_CHECKPOINT_DIR, "final_model.pth"),
    os.path.join(BASELINE_CHECKPOINT_DIR, "pretrained_generator.pth"),
]:
    if os.path.exists(candidate):
        ckpt = torch.load(candidate, map_location=device)
        if 'generator_state_dict' in ckpt:
            net_G.load_state_dict(ckpt['generator_state_dict'])
        else:
            net_G.load_state_dict(ckpt)
        print(f"loaded from {candidate}")
        break
else:
    print("no baseline checkpoint found - starting from scratch")

model = MainModel(net_G=net_G, lambda_L1=100., lambda_TV=1.0)


## Robustness Evaluation (Clean Model)

Test how PSNR and SSIM degrade as epsilon increases for FGSM and PGD.


In [ ]:
epsilons  = [0.0, 0.01, 0.02, 0.05]
num_batches_eval = 10  # number of validation batches to use for evaluation

results = {'fgsm': {}, 'pgd': {}}

model.net_G.eval()
for eps in epsilons:
    psnr_fgsm_list, ssim_fgsm_list = [], []
    psnr_pgd_list,  ssim_pgd_list  = [], []

    for i, data in enumerate(val_dl):
        if i >= num_batches_eval:
            break
        L  = data['L'].to(device)
        ab = data['ab'].to(device)

        with torch.no_grad():
            fake_clean = model.net_G(L)

        if eps > 0:
            L_fgsm = fgsm_attack(model, L, ab, epsilon=eps)
            L_pgd  = pgd_attack(model, L, ab, epsilon=eps)
        else:
            L_fgsm = L_pgd = L

        with torch.no_grad():
            fake_fgsm = model.net_G(L_fgsm)
            fake_pgd  = model.net_G(L_pgd)

        psnr_fgsm_list.append(compute_psnr(fake_fgsm, ab).item())
        ssim_fgsm_list.append(compute_ssim_batch(fake_fgsm, ab))
        psnr_pgd_list.append(compute_psnr(fake_pgd, ab).item())
        ssim_pgd_list.append(compute_ssim_batch(fake_pgd, ab))

    results['fgsm'][eps] = {'psnr': np.mean(psnr_fgsm_list), 'ssim': np.mean(ssim_fgsm_list)}
    results['pgd'][eps]  = {'psnr': np.mean(psnr_pgd_list),  'ssim': np.mean(ssim_pgd_list)}
    print(f"eps={eps:.2f}  fgsm psnr={results['fgsm'][eps]['psnr']:.2f}  pgd psnr={results['pgd'][eps]['psnr']:.2f}")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, metric in zip(axes, ['psnr', 'ssim']):
    fgsm_vals = [results['fgsm'][e][metric] for e in epsilons]
    pgd_vals  = [results['pgd'][e][metric]  for e in epsilons]
    ax.plot(epsilons, fgsm_vals, 'o-', label='FGSM')
    ax.plot(epsilons, pgd_vals,  's-', label='PGD')
    ax.set_xlabel('epsilon')
    ax.set_ylabel(metric.upper())
    ax.set_title(f'{metric.upper()} vs perturbation strength')
    ax.legend()

plt.tight_layout()
plt.savefig(os.path.join(ADV_CHECKPOINT_DIR, 'robustness_clean.png'), bbox_inches='tight')
plt.show()


## Adversarial Training

Retrain Stage 2 with adversarial examples mixed into each batch. At each iteration,
half the batch uses clean inputs and half uses FGSM-perturbed inputs.


In [ ]:
def train_adversarial(model, train_dl, val_dl, epochs, epsilon=0.02,
                      adv_checkpoint_dir=ADV_CHECKPOINT_DIR,
                      display_every=200, start_epoch=0):
    val_data = next(iter(val_dl))

    for e in range(start_epoch, epochs):
        meters = {k: AverageMeter() for k in
                  ['loss_D_fake', 'loss_D_real', 'loss_D',
                   'loss_G_GAN', 'loss_G_L1', 'loss_G_TV', 'loss_G']}
        i = 0

        for data in tqdm(train_dl, desc=f"epoch {e+1}/{epochs}"):
            L  = data['L'].to(device)
            ab = data['ab'].to(device)

            # mix clean and adversarial inputs
            n_adv = len(L) // 2
            if n_adv > 0:
                L_adv = fgsm_attack(model, L[:n_adv], ab[:n_adv], epsilon=epsilon)
                L_mixed = torch.cat([L_adv, L[n_adv:]], dim=0)
            else:
                L_mixed = L

            data_mixed = {'L': L_mixed, 'ab': ab}
            model.setup_input(data_mixed)
            model.optimize()

            for name, meter in meters.items():
                meter.update(getattr(model, name).item(), L.size(0))
            i += 1

            if i % display_every == 0:
                print(f"\nepoch {e+1}  iter {i}/{len(train_dl)}")
                for name, meter in meters.items():
                    print(f"  {name}: {meter.avg:.5f}")
                visualize(model, val_data, save=True,
                          path=os.path.join(adv_checkpoint_dir, f"epoch{e+1}_iter{i}.png"))

        if (e + 1) % 5 == 0 or (e + 1) == epochs:
            ckpt_path = os.path.join(adv_checkpoint_dir, f"checkpoint_epoch_{e+1}.pth")
            torch.save({
                'epoch': e + 1,
                'generator_state_dict':     model.net_G.state_dict(),
                'discriminator_state_dict': model.net_D.state_dict(),
                'optimizer_G_state_dict':   model.opt_G.state_dict(),
                'optimizer_D_state_dict':   model.opt_D.state_dict(),
            }, ckpt_path)
            print(f"  saved {ckpt_path}")

    torch.save(model.net_G.state_dict(),
               os.path.join(adv_checkpoint_dir, "adversarial_generator.pth"))
    print("adversarial training done")


# rebuild model from the baseline pretrained generator
net_G_adv = build_res_unet(n_input=1, n_output=2, size=SIZE)
pretrained = os.path.join(BASELINE_CHECKPOINT_DIR, "pretrained_generator.pth")
if os.path.exists(pretrained):
    net_G_adv.load_state_dict(torch.load(pretrained, map_location=device))
    print(f"loaded pretrained generator from {pretrained}")

model_adv = MainModel(net_G=net_G_adv, lambda_L1=100., lambda_TV=1.0)
train_adversarial(model_adv, train_dl, val_dl, epochs=20, epsilon=0.02)


## Compare: Standard vs Adversarially Trained Model


In [ ]:
model_adv.net_G.eval()
model.net_G.eval()

results_adv = {'standard': {}, 'adversarial': {}}
for eps in epsilons:
    for tag, mdl in [('standard', model), ('adversarial', model_adv)]:
        psnr_list, ssim_list = [], []
        for i, data in enumerate(val_dl):
            if i >= num_batches_eval:
                break
            L  = data['L'].to(device)
            ab = data['ab'].to(device)
            L_pgd = pgd_attack(mdl, L, ab, epsilon=eps) if eps > 0 else L
            with torch.no_grad():
                fake = mdl.net_G(L_pgd)
            psnr_list.append(compute_psnr(fake, ab).item())
            ssim_list.append(compute_ssim_batch(fake, ab))
        results_adv[tag][eps] = {'psnr': np.mean(psnr_list), 'ssim': np.mean(ssim_list)}

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, metric in zip(axes, ['psnr', 'ssim']):
    for tag, marker in [('standard', 'o-'), ('adversarial', 's--')]:
        vals = [results_adv[tag][e][metric] for e in epsilons]
        ax.plot(epsilons, vals, marker, label=tag)
    ax.set_xlabel('PGD epsilon')
    ax.set_ylabel(metric.upper())
    ax.set_title(f'{metric.upper()} under PGD attack')
    ax.legend()

plt.tight_layout()
plt.savefig(os.path.join(ADV_CHECKPOINT_DIR, 'robustness_comparison.png'), bbox_inches='tight')
plt.show()
